In [1]:
import torch, tqdm, time, os, json
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from models.est_model import GroundPressureModel
from torchmetrics import Accuracy
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import DeviceStatsMonitor, EarlyStopping
from sklearn.metrics import r2_score

In [2]:
monitor = DeviceStatsMonitor()

In [3]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [4]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [5]:
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403.json'), 'r') as file:
    hyper_params = json.load(file)

n_input = hyper_params['n_of_inputs']
h1_neuron = hyper_params['h1_neuron']
h2_neuron = hyper_params['h2_neuron']
n_output = hyper_params['n_of_outputs']

In [6]:
model = GroundPressureModel(n_input, h1_neuron, h2_neuron, n_output)
print(model)

GroundPressureModel(
  (basic_model): Sequential(
    (0): Linear(in_features=4, out_features=24, bias=True)
    (1): ReLU()
    (2): Linear(in_features=24, out_features=24, bias=True)
    (3): ReLU()
    (4): Linear(in_features=24, out_features=24, bias=True)
    (5): ReLU()
    (6): Linear(in_features=24, out_features=24, bias=True)
    (7): ReLU()
    (8): Linear(in_features=24, out_features=6, bias=True)
  )
  (shortcut_model): Sequential(
    (0): Linear(in_features=4, out_features=6, bias=True)
  )
  (loss_func): MSELoss()
)


In [7]:
data = pd.read_csv('./resources/sim_data.csv')
feature_names = ['lift_weight(ton)', 'lift_height(m)', 'rising_angle(deg)', 'swing_angle(deg)']
target_names = ['left_ground_pressure_min(kg/cm2)', 'left_ground_pressure_max(kg/cm2)', 'left_pressure_length(m)', 'right_ground_pressure_min(kg/cm2)', 'right_ground_pressure_max(kg/cm2)', 'right_pressure_length(m)']

feature_data = []
target_data = []

for feature_name in feature_names:
    feature_data.append(data[feature_name])

for target_name in target_names:
    target_data.append(data[target_name])

feature_data = np.array(feature_data, dtype=np.float32).T
target_data = np.array(target_data, dtype=np.float32).T

class TensorDataset(Dataset):
    def __init__(self, feature, target):
        self.x_data = torch.tensor(feature)
        self.y_data = torch.tensor(target)
        self.len = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

train_dataset = TensorDataset(feature_data, target_data)
train_data_loader = DataLoader(train_dataset, batch_size=len(train_dataset), num_workers=20)

/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/torch/utils/data/dataloader.py:561: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 16, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [8]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', verbose=True, min_delta=0.0001, patience=20)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='gpu', enable_progress_bar=True, max_epochs=10000)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [9]:
trainer.fit(model=model, train_dataloaders=train_data_loader)

/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/trainer/configuration_validator.py:72: PossibleUserWarning: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
  rank_zero_warn(
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type       | Params
----------------------------------------------
0 | basic_model    | Sequential | 870   
1 | shortcut_model | Sequential | 30    
2 | loss_func      | MSELoss    | 0     
----------------------------------------------
900       Trainable params
0         Non-trainable params
900       Total params
0.004     Total estimated model params size (MB)
/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/torch/utils/data/dataloader.py:561: UserWarning: This DataLoader will create 20 worker processes in total. Our suggested max number of worker in current system is 16, which is smaller than what this D

Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  1.10it/s, v_num=13]

Metric train_loss improved. New best score: 96.487


Epoch 1: 100%|██████████| 1/1 [00:00<00:00,  1.36it/s, v_num=13]

Metric train_loss improved by 23.996 >= min_delta = 0.0001. New best score: 72.490


...

Epoch 419: 100%|██████████| 1/1 [00:00<00:00,  1.44it/s, v_num=13]

Metric train_loss improved by 0.000 >= min_delta = 0.0001. New best score: 0.020


Epoch 439: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s, v_num=13]

Monitored metric train_loss did not improve in the last 20 records. Best score: 0.020. Signaling Trainer to stop.


Epoch 439: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s, v_num=13]


In [11]:
torch.save(model.state_dict(), './models/est_ground_pressure.pt')

In [10]:
model.eval()
for data in train_dataset:
    print(r2_score(data[1].detach().numpy() ,model(data[0]).detach().numpy()))
    print(model(data[0]).detach().numpy())
    print(data[1].detach().numpy())

0.9626875582830007
[ 4.4262705   0.06385374  3.4375947   4.588352   -0.07152462  3.4784465 ]
[3.97 0.   3.72 3.97 0.   3.72]
0.9870423123384626
[ 2.7706199  -0.12320054  3.9364955   4.9063587   0.12515745  3.7928882 ]
[2.61 0.   4.1  4.58 0.   4.1 ]
0.9929488996063904
[ 1.0091255  -0.13538027  5.697099    4.5474043   0.12162876  5.4750905 ]
[1.15 0.   5.59 4.13 0.   5.59]
0.9976199295738658
[ 3.1075795e+00  9.8776817e-04  5.0461926e+00  3.0904069e+00
 -3.5766125e-02  5.0890684e+00]
[2.93 0.   5.04 2.93 0.   5.04]
0.9894784165443863
[ 2.408513   -0.01806757  5.2390294   3.8194242   0.05914629  5.195538  ]
[2.16 0.   5.34 3.36 0.   5.34]
0.9965793358459694
[ 1.1769495  -0.0519526   6.5233183   3.4488816   0.22471929  6.2714114 ]
[1.25 0.   6.53 3.27 0.   6.53]
0.9983138921682613
[ 2.4592929e+00 -5.7293415e-02  6.5784416e+00  2.2553191e+00
 -5.6729317e-03  6.6967120e+00]
[2.21 0.   6.68 2.21 0.   6.68]
0.9932816153841271
[ 2.1658077  -0.06015587  6.880412    2.8744397   0.08454847  6.8183